# Search Algorithm Evaluation
This notebook computes NDCG@5, Precision@5, and MRR for the BM25, Dense, and Hybrid search algorithms using the manually graded `qrels.csv`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score
from search_engine import BM25Searcher, DenseSearcher, HybridSearcher

# --- CONFIGURATION ---
# Threshold for binary metrics (Precision/MRR). 
# Relevance >= THRESHOLD is treated as 1, else 0.
BINARY_THRESHOLD = 3
TOP_K = 5
# ---------------------

In [ ]:
# Load models
print("Loading dataframe...")
movies = pd.read_pickle('movies_with_embeddings.pkl')
bm25_searcher = BM25Searcher(movies)
dense_searcher = DenseSearcher(movies)
hybrid_searcher = HybridSearcher(bm25_searcher, dense_searcher)

In [ ]:
# Load qrels
try:
    qrels_df = pd.read_csv('qrels_to_grade.csv')
except FileNotFoundError:
    print("Error: qrels_to_grade.csv not found. Please run 02_pooling_generation.ipynb first.")
    raise

# Ensure relevance is numeric
qrels_df['relevance'] = pd.to_numeric(qrels_df['relevance'], errors='coerce').fillna(0)

# Build ground truth dictionary
ground_truth = {}
for _, row in qrels_df.iterrows():
    q = row['query']
    t = row['movie_title']
    r = float(row['relevance'])
    if q not in ground_truth:
        ground_truth[q] = {}
    ground_truth[q][t] = r

queries = list(ground_truth.keys())
print(f"Loaded ground truth for {len(queries)} queries.")

In [ ]:
def evaluate_method(searcher_func, queries, ground_truth, k=5, threshold=3):
    ndcg_scores = []
    mrr_scores = []
    p_at_k_scores = []
    
    for q in queries:
        results = searcher_func(q, top_k=k)
        
        # Calculate NDCG
        true_relevance = []
        predicted_scores = []
        
        for i, res in enumerate(results):
            title = res['title']
            rel = ground_truth[q].get(title, 0.0)
            true_relevance.append(rel)
            predicted_scores.append(res['score'])
            
        # Pad to k if results are less than k (rare but possible)
        while len(true_relevance) < k:
            true_relevance.append(0.0)
            predicted_scores.append(0.0)
            
        if np.sum(true_relevance) > 0:
            # scikit-learn expects 2D arrays
            ndcg = ndcg_score([true_relevance], [predicted_scores], k=k)
        else:
            ndcg = 0.0
        ndcg_scores.append(ndcg)
        
        # Calculate MRR and Precision@K
        binary_relevance = [1 if r >= threshold else 0 for r in true_relevance]
        p_at_k = np.sum(binary_relevance) / k
        p_at_k_scores.append(p_at_k)
        
        rr = 0.0
        for idx, val in enumerate(binary_relevance):
            if val == 1:
                rr = 1.0 / (idx + 1)
                break
        mrr_scores.append(rr)
        
    return {
        'NDCG@5': np.mean(ndcg_scores),
        'Precision@5': np.mean(p_at_k_scores),
        'MRR': np.mean(mrr_scores)
    }

# Evaluate all three
print("Evaluating BM25...")
bm25_metrics = evaluate_method(bm25_searcher.search, queries, ground_truth, TOP_K, BINARY_THRESHOLD)
print("Evaluating Dense...")
dense_metrics = evaluate_method(dense_searcher.search, queries, ground_truth, TOP_K, BINARY_THRESHOLD)
print("Evaluating Hybrid...")
hybrid_metrics = evaluate_method(hybrid_searcher.search, queries, ground_truth, TOP_K, BINARY_THRESHOLD)


In [ ]:
# Create results dataframe
results_df = pd.DataFrame([
    {'Method': 'BM25', **bm25_metrics},
    {'Method': 'Dense', **dense_metrics},
    {'Method': 'Hybrid', **hybrid_metrics}
])

print("\n=== FINAL EVALUATION RESULTS ===")
display(results_df.round(4))